# 10 - Stacking multimodal con ajuste de hiperparametros

Notebook de ensemble avanzado con StackingClassifier y busqueda exhaustiva sobre la capa meta.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import f1_score, classification_report


In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() and (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('No se pudo resolver notebooks_shiyi.')

PROJECT_ROOT = resolve_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
PRED_DIR = PROJECT_ROOT / 'predicciones_finales'
WEIGHTS_DIR = PROJECT_ROOT / 'weights_hpo'
REPORTS_DIR = PROJECT_ROOT / 'reports_hpo'
for d in [PRED_DIR, WEIGHTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FUSION_TRAIN_PATH = ARTIFACTS_DIR / 'fusion_features_train.npz'
FUSION_TEST_PATH = ARTIFACTS_DIR / 'fusion_features_test.npz'
required_paths = [FUSION_TRAIN_PATH, FUSION_TEST_PATH]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Missing split artifacts. Run 01_build_multimodal_features.ipynb first: '
        + ', '.join(missing_paths)
    )

train_bundle = np.load(FUSION_TRAIN_PATH, allow_pickle=True)
X_train_all = train_bundle['X_fusion'].astype(np.float32)
y_train_all = train_bundle['y'].astype(np.int64)
langs_train_all = train_bundle['langs'].astype(str)

test_bundle = np.load(FUSION_TEST_PATH, allow_pickle=True)
X_test_all = test_bundle['X_fusion'].astype(np.float32)
langs_test_all = test_bundle['langs'].astype(str)
sample_ids_test = test_bundle['sample_ids'].astype(str)

TEST_JSON_PATH = PROJECT_ROOT.parent / 'materials' / 'dataset_task3_exist2026' / 'test.json'
if not TEST_JSON_PATH.exists():
    raise FileNotFoundError(f'test.json not found: {TEST_JSON_PATH}')
with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_raw = json.load(f)
TEST_IDS = {str(k) for k in test_raw.keys()}
test_id_mask_test = np.array([sid in TEST_IDS for sid in sample_ids_test], dtype=bool)
print('X_train_all:', X_train_all.shape, '| etiquetas train validas:', int((y_train_all >= 0).sum()))
print('X_test_all:', X_test_all.shape, '| test rows:', int(test_id_mask_test.sum()))


In [ ]:
def build_stacking():
    base_estimators = [
        ('svm', Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(probability=True, class_weight='balanced', C=10.0, gamma='scale', random_state=42))
        ])),
        ('rf', RandomForestClassifier(
            n_estimators=600, max_depth=None, min_samples_leaf=1,
            class_weight='balanced_subsample', random_state=42, n_jobs=-1
        )),
        ('lr', Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=4000, class_weight='balanced', n_jobs=-1, random_state=42))
        ]))
    ]

    meta = LogisticRegression(max_iter=4000, class_weight='balanced', n_jobs=-1, random_state=42)
    stack = StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta,
        stack_method='predict_proba',
        cv=5,
        passthrough=False,
        n_jobs=-1
    )
    return stack


In [ ]:
def export_json(sample_ids, pred_binary, output_path):
    rows = [
        {'test_case': 'EXIST2026', 'id': str(sid), 'value': 'YES' if int(v) == 1 else 'NO'}
        for sid, v in zip(sample_ids, pred_binary)
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False)
    print('Exportado:', output_path, '| filas:', len(rows))

def run_stacking(tag, train_langs):
    mask = np.isin(langs_train_all, train_langs) & (y_train_all >= 0)
    X_train = X_train_all[mask]
    y_train = y_train_all[mask]

    if len(np.unique(y_train)) < 2:
        raise RuntimeError(f'{tag}: subset sin dos clases.')

    stack = build_stacking()
    param_grid = {
        'final_estimator__C': [0.1, 1.0, 5.0, 10.0],
        'passthrough': [False, True]
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(
        estimator=stack,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    gs.fit(X_train, y_train)

    print(tag, 'best CV f1_macro:', gs.best_score_)
    print('best params:', gs.best_params_)

    cv_rows = []
    for i in range(len(gs.cv_results_['params'])):
        cv_rows.append({
            'config': tag,
            'mean_test_score': float(gs.cv_results_['mean_test_score'][i]),
            'std_test_score': float(gs.cv_results_['std_test_score'][i]),
            **{f'param_{k}': v for k, v in gs.cv_results_['params'][i].items()}
        })
    cv_df = pd.DataFrame(cv_rows).sort_values('mean_test_score', ascending=False)
    cv_path = REPORTS_DIR / f'stacking_cv_{tag}.csv'
    cv_df.to_csv(cv_path, index=False)
    print('Reporte CV:', cv_path)

    best_model = gs.best_estimator_
    pred_train = best_model.predict(X_train)
    print('Train macro-f1:', f1_score(y_train, pred_train, average='macro'))
    print(classification_report(y_train, pred_train, digits=4))

    model_path = WEIGHTS_DIR / f'stacking_best_{tag}.joblib'
    joblib.dump(best_model, model_path)
    print('Modelo guardado:', model_path)

    infer_mask = test_id_mask_test & np.isin(langs_test_all, train_langs)
    if not infer_mask.any():
        raise RuntimeError(f'{tag}: no test rows for selected langs in fusion_features_test.npz')
    pred_test = best_model.predict(X_test_all[infer_mask])
    out_path = PRED_DIR / f'BeingChillingWeWillWin_STACK_{tag}.json'
    export_json(sample_ids_test[infer_mask], pred_test, out_path)

run_stacking('ES', ['es'])
